## Wybór warzyw do modelu

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder



In [2]:
data_path = 'Data'

for split in ['train', 'test', 'val']:
    split_path = os.path.join(data_path, split)
    folders = [name for name in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, name))]
    df_folders = pd.DataFrame({'Nazwa folderu': sorted(folders)})
    df_folders.index += 1

In [ ]:
dataset_path = 'Data'
split = 'train'
split_path = os.path.join(dataset_path, split)

class_names = sorted([
    name for name in os.listdir(split_path) 
    if os.path.isdir(os.path.join(split_path, name))
])

data = []
labels = []

for class_name in class_names:
    class_dir = os.path.join(split_path, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (32, 32))
        data.append(img)
        labels.append(class_names.index(class_name))

X_train = np.array(data)
y_train = np.array(labels)

X_train = X_train.reshape(X_train.shape[0], -1)

In [ ]:
X_train.shape

(8120, 3072)

In [ ]:
dataset_path = 'Data'
split = 'test'
split_path = os.path.join(dataset_path, split)

class_names = sorted([
    name for name in os.listdir(split_path) 
    if os.path.isdir(os.path.join(split_path, name))
])

data = []
labels = []

for class_name in class_names:
    class_dir = os.path.join(split_path, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (32, 32))
        data.append(img)
        labels.append(class_names.index(class_name))

X_test = np.array(data)
y_test = np.array(labels)

X_test = X_test.reshape(X_test.shape[0], -1)

In [ ]:
X_test.shape

(1740, 3072)

In [ ]:
model = SVC(kernel='linear')

model.fit(X_train, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [ ]:
y_pred=model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred, target_names=folders))

                                               precision    recall  f1-score   support

                              Healthy (Mango)       0.86      0.85      0.86        60
                        Early blight (Tomato)       0.69      0.78      0.73        60
       Tomato Yellow Leaf Curl Virus (Tomato)       0.38      0.42      0.40        60
                     Bacterial Canker (Mango)       0.56      0.83      0.67        60
                               Healthy (Rice)       0.67      0.75      0.71        60
                Bacterial Leaf Spot (Pumpkin)       0.98      0.90      0.94        60
                         Target Spot (Tomato)       0.85      0.95      0.90        60
                           Gall Midge (Mango)       0.64      0.53      0.58        60
                       Downy Mildew (Pumpkin)       0.53      0.57      0.55        60
                       Powdery Mildew (Mango)       0.50      0.58      0.54        60
                             LeafBlast (Ri

In [ ]:
import re

report_dict = classification_report(y_test, y_pred, target_names=sorted(folders), output_dict=True)

rows = []
for class_name, metrics in report_dict.items():
    if class_name in ("accuracy", "macro avg", "weighted avg"):
        continue
    match = re.search(r"\((.+?)\)", class_name)
    plant = match.group(1) if match else "Unknown Disease"
    rows.append({
        "Klasa": class_name,
        "Roslina": plant,
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1-score": metrics["f1-score"],
        "Support": int(metrics["support"])
    })

df = pd.DataFrame(rows)

summary = (
    df.groupby("Roslina")
    .agg(
        Precision=("Precision", "mean"),
        Recall=("Recall", "mean"),
        F1_score=("F1-score", "mean"),
        Support=("Support", "sum"),
        Liczba_klas=("Klasa", "count")
    )
    .round(4)
    .sort_values("F1_score", ascending=False)
    .reset_index()
)
summary.index += 1
summary

,Roslina,Precision,Recall,F1_score,Support,Liczba_klas
1,Strawberry,0.8436,0.8333,0.8379,120,2
2,Mango,0.7539,0.7792,0.7649,480,8
3,Tomato,0.6500,0.6167,0.6261,600,10
4,Pumpkin,0.5834,0.5767,0.5779,300,5
5,Rice,0.4243,0.4333,0.4271,240,4


## Wyniki ewaluacji modelu

### Ogólne metryki

| Metryka | Wartość |
|---|---|
| Dokładność (Accuracy) | **64%** |
| Macro avg Precision | 0.65 |
| Macro avg Recall | 0.64 |
| Macro avg F1-score | 0.64 |
| Liczba próbek testowych | 1740 (29 klas × 60) |

---

### Wyniki według rośliny

| Roślina | Precision | Recall | F1-score | Liczba klas |
|---|---|---|---|---|
| Strawberry | 0.844 | 0.833 | 0.838 | 2 |
| Mango | 0.754 | 0.779 | 0.765 | 8 |
| Tomato | 0.650 | 0.617 | 0.626 | 10 |
| Pumpkin | 0.583 | 0.577 | 0.578 | 5 |
| Rice | 0.424 | 0.433 | 0.427 | 4 |

---

### Najlepiej rozpoznawane klasy (F1 ≥ 0.80)

| Klasa | Precision | Recall | F1-score |
|---|---|---|---|
| Bacterial Leaf Spot (Pumpkin) | 0.98 | 0.90 | **0.94** |
| Healthy Leaf (Pumpkin) | 0.92 | 0.95 | **0.93** |
| Target Spot (Tomato) | 0.85 | 0.95 | **0.90** |
| Healthy (Mango) | 0.86 | 0.85 | **0.86** |
| LeafBlast (Rice) | 0.81 | 0.83 | **0.82** |
| Anthracnose (Mango) | 0.88 | 0.75 | **0.81** |
| Leaf Mold (Tomato) | 0.82 | 0.77 | **0.79** |

---

### Klasy wymagające poprawy (F1 < 0.50)

| Klasa | Precision | Recall | F1-score | Możliwa przyczyna |
|---|---|---|---|---|
| Hispa (Rice) | 0.30 | 0.23 | **0.26** | Wizualne podobieństwo do innych klas ryżu |
| Leaf scorch (Strawberry) | 0.35 | 0.38 | **0.37** | Mała liczba klas truskawki, brak kontrastu |
| healthy (Tomato) | 0.38 | 0.37 | **0.37** | Mylona z innymi zdrowymi liśćmi |
| Tomato Yellow Leaf Curl Virus | 0.38 | 0.42 | **0.40** | Subtelne objawy wizualne |

---

### Wnioski

- Model najlepiej radzi sobie z **Pumpkin** i **Strawberry** — mała liczba klas ułatwia klasyfikację.
- Największym wyzwaniem jest **Rice** (4 klasy, F1 = 0.43) — klasy są do siebie wizualnie podobne.
- **Tomato** (10 klas, F1 = 0.63) — największa złożoność ze względu na największą liczbę klas.
- Ogólna dokładność 64% przy 29 klasach z bardzo zróżnicowanymi roślinami jest wynikiem bazowym — rekomendowane kolejne kroki to fine-tuning, augmentacja danych lub osobne głowice klasyfikacyjne per roślina.
